## 训练模型调用mac上的GPU该如何实现

在 macOS 上，PyTorch 目前仅支持通过 Apple Silicon（M1/M2 芯片）使用 GPU（即 Metal 后端），不支持 NVIDIA GPU（macOS 下没有 CUDA 驱动）。如果你使用的是 Apple Silicon，可以通过以下步骤启用 GPU 加速：

安装支持 MPS（Metal Performance Shaders）的 PyTorch 版本。可参考 PyTorch 官网的安装说明，选择 macOS + Conda + Python + MPS。
在代码中检测并使用 MPS 设备：

In [3]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 定义一个简单的模型
class MyModel(torch.nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.linear = torch.nn.Linear(200000, 1)

    def forward(self, x):
        return self.linear(x)
    
# 创建一个随机输入张量
X = torch.randn(5, 200000)

# 示例：将模型和数据移动到 MPS
model = MyModel().to(device)
X = X.to(device)

# 执行前向传播
output = model(X)
# 打印输出
print(output)

tensor([[-0.6903],
        [ 0.1633],
        [-0.0061],
        [-0.4123],
        [-0.0215]], device='mps:0', grad_fn=<LinearBackward0>)


是的，输出中的 device='mps:0' 说明你的张量和模型已经在 Mac 的 GPU（通过 Apple 的 Metal Performance Shaders，简称 MPS）上进行计算。这表明你已经成功调用了 Mac 上的 GPU 进行模型的前向传播（推理或训练时都一样）。

只要你的模型和数据都 .to(device)，且 device 是 mps，就会用 Mac 的 GPU 运算。
注意：MPS 只支持 Apple Silicon（M1/M2）系列芯片的 GPU。

## 查询张量所在设备

In [4]:
Y = torch.randn(5, 1)
Y.device

device(type='cpu')

In [5]:
Y = torch.randn(5, 1).to(device)
Y.device

device(type='mps', index=0)

In [8]:
model.linear.weight.data.device

device(type='mps', index=0)